# Campaign Deployment Overlay Example

This notebook loads the stored configuration-conditional calibration artifact, applies it to every sensor in one selected deployment campaign, and renders only the final cross-node before/after PSD overlay animation.

The animation title keeps the campaign label, aligned record number, and the mean pairwise RMSE before and after calibration so same-scene agreement can be inspected frame by frame.


In [19]:
# Shared workflow configuration lives under ``config/notebook_workflow/``.
# Edit those files instead of maintaining notebook-local campaign lists.
NOTEBOOK_WORKFLOW_CONFIG_DIRNAME = "config/notebook_workflow"

# Optional notebook-local deployment override within the configured testing split.
DEPLOYMENT_CAMPAIGN_LABEL = None


In [20]:
from __future__ import annotations

from pathlib import Path
import shutil
import sys
from tempfile import mkdtemp

from IPython.display import HTML
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import numpy as np


def resolve_repo_root() -> Path:
    """Return the repository root regardless of the current notebook cwd."""

    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "measurement_calibration").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not resolve the repository root from the notebook cwd")


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from measurement_calibration import (
    DEFAULT_ARCHIVED_ARTIFACTS_DIR,
    DEFAULT_CAMPAIGNS_DATA_DIR,
    DEFAULT_PRODUCTION_ARTIFACT_DIR,
    DEFAULT_PRODUCTION_PARAMETERS_FILENAME,
    DeploymentCalibrationResult,
    FileSystemCampaignSensorDataRepository,
    FrequencyBasisConfig,
    PersistentModelConfig,
    TwoLevelFitConfig,
    archive_artifact_directory,
    build_cross_node_campaign_animation_data,
    calibrate_sensor_observations,
    fit_and_save_calibration_corpus_model,
    format_cross_node_overlay_title,
    load_notebook_workflow_config,
    load_two_level_calibration_artifact,
    prepare_calibration_campaign,
    prepare_calibration_corpus,
    resolve_cross_node_overlay_limits_db,
    resolve_global_excluded_sensor_ids_by_campaign,
)

CAMPAIGNS_ROOT = REPO_ROOT / DEFAULT_CAMPAIGNS_DATA_DIR
PRODUCTION_MODEL_DIR = REPO_ROOT / DEFAULT_PRODUCTION_ARTIFACT_DIR
ARCHIVE_MODELS_ROOT = REPO_ROOT / DEFAULT_ARCHIVED_ARTIFACTS_DIR
PRODUCTION_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
WORKFLOW_CONFIG = load_notebook_workflow_config(
    REPO_ROOT / NOTEBOOK_WORKFLOW_CONFIG_DIRNAME
)
TRAINING_CAMPAIGN_LABELS = WORKFLOW_CONFIG.training_campaign_labels
TESTING_CAMPAIGN_LABELS = WORKFLOW_CONFIG.testing_campaign_labels
WORKFLOW_CAMPAIGN_LABELS = WORKFLOW_CONFIG.workflow_campaign_labels
EXCLUDED_NODES = WORKFLOW_CONFIG.excluded_sensor_ids

CAMPAIGN_REPOSITORY = FileSystemCampaignSensorDataRepository(
    campaigns_root=CAMPAIGNS_ROOT
)
AVAILABLE_CAMPAIGN_LABELS = CAMPAIGN_REPOSITORY.list_campaign_labels()
missing_campaign_labels = sorted(
    set(WORKFLOW_CAMPAIGN_LABELS).difference(AVAILABLE_CAMPAIGN_LABELS)
)
if missing_campaign_labels:
    raise ValueError(
        "Notebook workflow configuration references campaigns that do not exist "
        f"under {CAMPAIGNS_ROOT}: {missing_campaign_labels}"
    )

GLOBAL_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN = resolve_global_excluded_sensor_ids_by_campaign(
    campaign_labels=WORKFLOW_CAMPAIGN_LABELS,
    excluded_sensor_ids=EXCLUDED_NODES,
    campaigns_root=CAMPAIGNS_ROOT,
)
TRAINING_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN = {
    campaign_label: GLOBAL_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN[campaign_label]
    for campaign_label in TRAINING_CAMPAIGN_LABELS
}
TESTING_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN = {
    campaign_label: GLOBAL_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN[campaign_label]
    for campaign_label in TESTING_CAMPAIGN_LABELS
}
if DEPLOYMENT_CAMPAIGN_LABEL is None:
    DEPLOYMENT_CAMPAIGN_LABEL = TESTING_CAMPAIGN_LABELS[0]
elif DEPLOYMENT_CAMPAIGN_LABEL not in TESTING_CAMPAIGN_LABELS:
    raise ValueError(
        "DEPLOYMENT_CAMPAIGN_LABEL must belong to the configured testing split, "
        f"got {DEPLOYMENT_CAMPAIGN_LABEL!r} not in {TESTING_CAMPAIGN_LABELS}"
    )

BASIS_CONFIG = FrequencyBasisConfig(
    n_gain_basis=10,
    n_floor_basis=8,
    n_variance_basis=8,
    spline_degree=3,
)
MODEL_CONFIG = PersistentModelConfig(
    sensor_embedding_dim=4,
    configuration_latent_dim=4,
)
FIT_CONFIG = TwoLevelFitConfig(
    n_outer_iterations=8,
    n_gradient_steps=20,
    learning_rate=0.03,
    sigma_min=1.0e-8,
    adaptive_variance_floor_ratio=1.0e-4,
    select_best_outer_iterate=True,
    early_stopping_patience=5,
    divergence_tolerance_ratio=10.0,
    refresh_campaign_variance_from_residuals=True,
    variance_refresh_ridge=1.0e-6,
)


## Load Or Build The Stored Model

The deployment notebook stays self-contained: if `models/production/` does
not contain a promoted artifact yet, the notebook fits the same lightweight
training configuration, stores the new production bundle, and then loads it.


In [21]:
if not (PRODUCTION_MODEL_DIR / "manifest.json").exists():
    staging_output_dir = Path(
        mkdtemp(prefix=".production_staging__", dir=PRODUCTION_MODEL_DIR.parent)
    )
    try:
        preparation = prepare_calibration_corpus(
            campaign_labels=TRAINING_CAMPAIGN_LABELS,
            campaigns_root=CAMPAIGNS_ROOT,
            excluded_sensor_ids_by_campaign=TRAINING_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN,
        )
        fit_and_save_calibration_corpus_model(
            preparation=preparation,
            output_dir=staging_output_dir,
            basis_config=BASIS_CONFIG,
            model_config=MODEL_CONFIG,
            fit_config=FIT_CONFIG,
            parameters_filename=DEFAULT_PRODUCTION_PARAMETERS_FILENAME,
        )
        archive_artifact_directory(
            output_dir=PRODUCTION_MODEL_DIR,
            archive_root=ARCHIVE_MODELS_ROOT,
            archive_label="production",
        )
        if PRODUCTION_MODEL_DIR.exists():
            PRODUCTION_MODEL_DIR.rmdir()
        staging_output_dir.rename(PRODUCTION_MODEL_DIR)
    except Exception:
        shutil.rmtree(staging_output_dir, ignore_errors=True)
        raise

artifact = load_two_level_calibration_artifact(PRODUCTION_MODEL_DIR)
deployment_excluded_sensor_ids = TESTING_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN[
    DEPLOYMENT_CAMPAIGN_LABEL
]
deployment_campaign = prepare_calibration_campaign(
    campaign_label=DEPLOYMENT_CAMPAIGN_LABEL,
    campaigns_root=CAMPAIGNS_ROOT,
    excluded_sensor_ids=deployment_excluded_sensor_ids,
)


## Cross-Node Overlay Animation

This notebook intentionally emits only the final before/after campaign overlay animation for the selected deployment campaign.

Each frame overlays every retained sensor on the full frequency grid, with the title showing the campaign label, aligned record number, and the mean pairwise RMSE before and after calibration.


In [22]:
def calibrate_campaign_sensor(
    sensor_id: str,  # Sensor to calibrate
    observations_power: np.ndarray,  # Sensor PSD tensor [linear power]
) -> DeploymentCalibrationResult:  # Calibrated deployment output
    """Bind the deployment artifact and campaign metadata to one sensor."""

    return calibrate_sensor_observations(
        result=artifact.result,
        sensor_id=sensor_id,
        configuration=deployment_campaign.campaign.configuration,
        frequency_hz=deployment_campaign.campaign.frequency_hz,
        observations_power=observations_power,
    )


animation_data = build_cross_node_campaign_animation_data(
    campaign=deployment_campaign.campaign,
    calibrate_sensor=calibrate_campaign_sensor,
)
frequency_mhz = animation_data.frequency_hz / 1.0e6
y_min_db, y_max_db = resolve_cross_node_overlay_limits_db(animation_data)

fig, (raw_axis, calibrated_axis) = plt.subplots(
    2,
    1,
    figsize=(14, 8.5),
    sharex=True,
    constrained_layout=True,
)

line_handles = {}
color_map = plt.get_cmap("tab10")
for sensor_index, sensor_id in enumerate(animation_data.sensor_ids):
    color = color_map(sensor_index % color_map.N)
    raw_line, = raw_axis.plot([], [], linewidth=1.0, alpha=0.9, color=color, label=sensor_id)
    calibrated_line, = calibrated_axis.plot(
        [],
        [],
        linewidth=1.0,
        alpha=0.9,
        color=color,
        label=sensor_id,
    )
    line_handles[sensor_id] = (raw_line, calibrated_line)

# Keep both panels on the same scale so the visual compression is
# directly comparable before and after calibration.
for axis, axis_title in (
    (raw_axis, "Before Calibration"),
    (calibrated_axis, "After Calibration"),
):
    axis.set_xlim(frequency_mhz[0], frequency_mhz[-1])
    axis.set_ylim(y_min_db, y_max_db)
    axis.set_ylabel("Power [dB]")
    axis.set_title(axis_title)
    axis.grid(alpha=0.3)

calibrated_axis.set_xlabel("Frequency [MHz]")
raw_axis.legend(loc="upper right", ncols=min(4, animation_data.n_sensors))


def update_animation(record_index: int):
    """Update both overlay panels for one aligned record."""

    artists = []
    for sensor_index, sensor_id in enumerate(animation_data.sensor_ids):
        raw_line, calibrated_line = line_handles[sensor_id]
        raw_line.set_data(
            frequency_mhz,
            animation_data.raw_power_db[sensor_index, record_index],
        )
        calibrated_line.set_data(
            frequency_mhz,
            animation_data.calibrated_power_db[sensor_index, record_index],
        )
        artists.extend((raw_line, calibrated_line))

    fig.suptitle(
        format_cross_node_overlay_title(animation_data, record_index),
        fontsize=14,
    )
    return artists


animation = FuncAnimation(
    fig,
    update_animation,
    frames=animation_data.n_records,
    interval=250,
    blit=False,
    cache_frame_data=False,
)
plt.close(fig)
HTML(animation.to_jshtml(default_mode="loop"))
